In [1]:
from kafka import KafkaConsumer
import json
import matplotlib.pyplot as plt
import time

In [2]:
# Initialize data storage for plotting
timestamps = []
water_temperatures = []
ph_levels = []
turbidities = []
dissolved_oxygen_levels = []

In [3]:
# Kafka configuration
def initialize_consumer():
    kafka_topic = "water_quality"
    kafka_bootstrap_servers = ["localhost:9092"]

    # Create Kafka consumer
    consumer = KafkaConsumer(
        kafka_topic,
        bootstrap_servers=kafka_bootstrap_servers,
        value_deserializer=lambda m: json.loads(m.decode('utf-8')),
        auto_offset_reset='latest',
        enable_auto_commit=True,
        group_id="water_quality_processors"
        )
    return consumer

In [4]:
# Receive all published messages and update plot
def update_plot(consumer):
    try:
        for message in consumer:
            # Parse the message
            sensor_data = message.value
            sensor_id = message.key
            print(f"Received: sensor_id:{sensor_id},{sensor_data}")

            # Update data storage
            timestamps.append(sensor_data['timestamp'])
            water_temperatures.append(sensor_data['water_temperature'])
            ph_levels.append(sensor_data['ph_level'])
            turbidities.append(sensor_data['turbidity'])
            dissolved_oxygen_levels.append(sensor_data['dissolved_oxygen'])

            # Keep only the last 100 entries for plotting
            if len(timestamps) > 100:
                timestamps.pop(0)
                water_temperatures.pop(0)
                ph_levels.pop(0)
                turbidities.pop(0)
                dissolved_oxygen_levels.pop(0)

            # Clear the current axes and redraw the plots
            plt.figure(figsize=(10, 8))

            plt.subplot(2, 2, 1)
            plt.plot(timestamps, water_temperatures, label="Water Temperature", color="blue")
            plt.title("Water Temperature")
            plt.ylabel("°C")

            plt.subplot(2, 2, 2)
            plt.plot(timestamps, ph_levels, label="pH Level", color="green")
            plt.title("pH Level")
            plt.ylabel("pH")

            plt.subplot(2, 2, 3)
            plt.plot(timestamps, turbidities, label="Turbidity", color="orange")
            plt.title("Turbidity")
            plt.ylabel("NTU")

            plt.subplot(2, 2, 4)
            plt.plot(timestamps, dissolved_oxygen_levels, label="Dissolved Oxygen", color="red")
            plt.title("Dissolved Oxygen")
            plt.ylabel("mg/L")

            plt.tight_layout()

            # Save the plot as an image
            plt.savefig(f"water_quality_plot{sensor_id}.png")
            plt.close()

            break  # Process one message at a time
    except KeyboardInterrupt:
        print("Stopped consuming messages.")
        consumer.close()

In [ ]:
consumer = initialize_consumer()
print("Subscribed to Kafka topic 'water_quality'.")

try:
    while True:
        update_plot(consumer)
except KeyboardInterrupt:
    print("Stopped visualization.")
    consumer.close()

Subscribed to Kafka topic 'water_quality'.
Received: sensor_id:b'4',{'timestamp': 1772380458, 'water_temperature': 29.056009822172637, 'ph_level': 8.58122364684338, 'turbidity': 21.08, 'dissolved_oxygen': 6.08}
Received: sensor_id:b'3',{'timestamp': 1772380458, 'water_temperature': 27.535877663441497, 'ph_level': 8.491354879900813, 'turbidity': 5.38, 'dissolved_oxygen': 9.31}
Received: sensor_id:b'1',{'timestamp': 1772380458, 'water_temperature': 27.47295157335033, 'ph_level': 7.916441828829642, 'turbidity': 45.41, 'dissolved_oxygen': 11.12}
Received: sensor_id:b'1',{'timestamp': 1772380459, 'water_temperature': 28.129857973886487, 'ph_level': 8.063399318132557, 'turbidity': 42.33, 'dissolved_oxygen': 9.88}
Received: sensor_id:b'4',{'timestamp': 1772380459, 'water_temperature': 27.28134346100356, 'ph_level': 7.911304984442733, 'turbidity': 25.88, 'dissolved_oxygen': 6.61}
Received: sensor_id:b'3',{'timestamp': 1772380459, 'water_temperature': 28.16033169730968, 'ph_level': 8.3829211413